In [ ]:
import shutil
from langchain_core.documents import Document

def generate_question(state: InterviewState) -> InterviewState:

    """
      질문 생성 (고도화 버전)
        - Chroma Vector DB에서 유사 질문 3개를 검색하여 LLM 프롬프트에 참고로 포함

    """

    # LLM 및 임베딩 모델 정의
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")

    # 유사 질문 생성 프롬프트
    sub_prompt = ChatPromptTemplate.from_template("""
    당신은 전문 면접관입니다. 아래 대화 맥락을 참고하여,
    현재 질문과 유사한 의도나 주제를 가진 면접 질문 3개를 생성하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 최근 질문 ---
    {current_question}

    --- 현재 질문 전략 ---
    {current_strategy}

    --- 지원자 답변 ---
    {current_answer}

    생성 규칙:
    1. 각 질문은 현재 질문과 **의도, 주제, 평가 관점**이 유사해야 합니다.
    2. 문장 구조나 표현은 다양하게 바꿔서 작성하세요. (동어 반복 금지)
    3. 한 문장으로 작성하고, 자연스럽고 명확한 한국어 질문 형태로 만드세요.
    4. 어조는 전문적이며 격식체를 유지합니다.
    5. 모든 질문은 30자 이내로 간결하게 작성하세요.
    6. 강조기호(** 등)나 이모티콘은 사용하지 마세요.

    출력 형식:
    1) 질문 1
    2) 질문 2
    3) 질문 3
    """)

    response = (sub_prompt | llm).invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "current_strategy": state.get("current_strategy","")

    })

    # 결과 파싱
    lines = [line.strip("123). ").strip() for line in response.content.split("\n") if line.strip()]
    similar_questions = [q for q in lines if q and len(q) > 3][:3]


    # 벡터 DB 생성
    docs = [Document(page_content=q) for q in similar_questions]
    vectorstore = Chroma.from_documents(docs, embedding, persist_directory=None)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 최종 질문 생성

    similar_docs = retriever.invoke(state.get("current_answer", ""))
    similar_texts = "\n".join([f"- {doc.page_content}" for doc in similar_docs])

    main_prompt = ChatPromptTemplate.from_template("""
    당신은 면접관입니다. 아래 정보를 기반으로 지원자에게 추가로 물어볼 질문을 생성하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 질문 전략 ---
    {question_strategy}

    --- 현재 질문 전략 ---
    {current_strategy}

    --- 이전 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    --- 답변 평가 ---
    {evaluation}

    --- 참고용 유사 질문 ---
    {similar_examples}

    생성 규칙:
    1. 지원자의 답변 내용을 기반으로 핵심 근거, 구체적 행동, 결과를 더 깊이 탐색하는 심화 질문을 생성합니다.
    2. 새로운 주제로 전환하지 말고, 반드시 이전 질문의 맥락을 유지하세요.
    3. 한 문장으로 자연스럽고 명확한 질문을 작성합니다.
    4. 어조는 전문적이며 격식체를 유지합니다.
    5. 질문은 30자 이내로 간결하게 작성하세요.
    6. 강조기호(** 등)나 이모티콘은 사용하지 마세요.

    """)


    chain = main_prompt | llm
    response = chain.invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "question_strategy": state.get("question_strategy", {}),
        "current_strategy" : state.get("current_strategy",""),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "evaluation": state.get("evaluation", []),
        "next_step": state.get("next_step",""),
        "similar_examples": similar_texts
    })

    next_question = response.content.strip()

    return {
        **state,
        "current_question": next_question,
        "current_answer": "",
    }
